In [2]:
import os, gc, math, time, random, warnings, glob
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from datasets import load_dataset, interleave_datasets

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__)
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.float16   # T4 pe bf16 nahi hai
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
print("AMP:", USE_AMP, "| dtype:", AMP_DTYPE)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

BASE_DIR   = "/content/drive/MyDrive/gpt100m"
CHUNK_DIR  = f"{BASE_DIR}/chunks"
CKPT_DIR   = f"{BASE_DIR}/checkpoints"
os.makedirs(CHUNK_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

TOK_SAMPLE  = f"{BASE_DIR}/tok_sample.txt"
TOK_PATH    = f"{BASE_DIR}/tokenizer_32k.json"
#LATEST_PATH = f"{CKPT_DIR}/latest.pth"
LATEST_PATH = "/content/drive/MyDrive/gpt100m/checkpoints/latest.pth"
BEST_PATH   = f"{CKPT_DIR}/best.pth"

print("\nBase:", BASE_DIR)
print("CKPT files:", os.listdir(CKPT_DIR))
print("Chunks:", len(glob.glob(f"{CHUNK_DIR}/chunk_*.pt")))

Torch: 2.11.0+cu130
Device: cuda
GPU: Tesla T4
VRAM: 15.64 GB
AMP: True | dtype: torch.float16

Base: /content/drive/MyDrive/gpt100m
CKPT files: ['best.pth.bak', 'best.pth', 'latest.pth', 'latest.pth.bak']
Chunks: 100


In [ ]:
# =========================================================
# CELL 2: Build 500MB sample for tokenizer training
# =========================================================
if not os.path.exists(TOK_SAMPLE):
    print("Building 500MB sample...")
    ds_tok = interleave_datasets([
        load_dataset("HuggingFaceFW/fineweb-edu", "sample-10BT", split="train", streaming=True),
        load_dataset("roneneldan/TinyStories", split="train", streaming=True),
    ], probabilities=[0.8, 0.2], seed=SEED)

    total_bytes = 0
    with open(TOK_SAMPLE, "w", encoding="utf-8") as f:
        for ex in ds_tok:
            t = ex["text"]
            f.write(t + "\n")
            total_bytes += len(t.encode("utf-8")) + 1
            if total_bytes % 50_000_000 < 5000:
                print(f"  {total_bytes/1e6:.0f} MB")
            if total_bytes >= 500_000_000:
                break
    print(f"Done: {total_bytes/1e6:.0f} MB")
else:
    print(f"Sample exists: {os.path.getsize(TOK_SAMPLE)/1e6:.0f} MB")

Building 500MB sample...


README.md:   0%|          | 0.00/26.4k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

  0 MB
  100 MB
  150 MB
  150 MB
  200 MB
  300 MB
  350 MB
  450 MB
  450 MB
  500 MB
Done: 500 MB


In [3]:
# =========================================================
# CELL 3: Train BPE tokenizer (ByteLevel, 32K vocab)
# =========================================================
if not os.path.exists(TOK_PATH):
    print("Training tokenizer...")
    tok = Tokenizer(models.BPE(unk_token="<unk>"))
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder       = decoders.ByteLevel()

    trainer = trainers.BpeTrainer(
        vocab_size=32000,
        min_frequency=2,
        special_tokens=["<unk>", "<pad>", "<eos>", "<bos>",
                        "<|user|>", "<|assistant|>", "<|end|>"],
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        show_progress=True,
    )
    tok.train([TOK_SAMPLE], trainer)
    tok.save(TOK_PATH)
    print("Tokenizer saved")
else:
    print("Tokenizer exists")

tokenizer = Tokenizer.from_file(TOK_PATH)
VOCAB_SIZE = tokenizer.get_vocab_size()
END_ID     = tokenizer.token_to_id("<|end|>")
PAD_ID     = tokenizer.token_to_id("<pad>")

print("Vocab:", VOCAB_SIZE)

test_str = 'It said, "Don\'t touch the key."'
enc = tokenizer.encode(test_str)
print("\nTokens:", enc.tokens)
print("Decoded:", tokenizer.decode(enc.ids))
print("Match:", tokenizer.decode(enc.ids) == test_str)

Tokenizer exists
Vocab: 32000

Tokens: ['It', 'Ġsaid', ',', 'Ġ"', 'Don', "'t", 'Ġtouch', 'Ġthe', 'Ġkey', '."']
Decoded: It said, "Don't touch the key."
Match: True


In [4]:
# =========================================================
# CELL 4: Tokenize full dataset → chunks (resumable)
# int16 storage = ~2 bytes/token → ~7GB for 3.5B tokens
# =========================================================
BLOCK_SIZE         = 1024
EXAMPLES_PER_CHUNK = 50_000
MAX_CHUNKS         = 100   # ~3.5B tokens

existing_chunks = sorted(glob.glob(f"{CHUNK_DIR}/chunk_*.pt"))
print(f"Existing: {len(existing_chunks)} | Target: {MAX_CHUNKS}")

if len(existing_chunks) < MAX_CHUNKS:
    ds_tok = interleave_datasets([
        load_dataset("HuggingFaceFW/fineweb-edu", "sample-10BT", split="train", streaming=True),
        load_dataset("roneneldan/TinyStories", split="train", streaming=True),
    ], probabilities=[0.8, 0.2], seed=SEED)

    skip_n = len(existing_chunks) * EXAMPLES_PER_CHUNK
    if skip_n > 0:
        print(f"Skipping {skip_n:,} examples (1-2 min)...")
        ds_tok = ds_tok.skip(skip_n)

    buffer = []
    chunk_idx = len(existing_chunks)
    examples_done = skip_n

    for ex in ds_tok:
        ids = tokenizer.encode(ex["text"]).ids
        buffer.extend(ids)
        buffer.append(END_ID)
        examples_done += 1

        if examples_done % EXAMPLES_PER_CHUNK == 0:
            arr = torch.tensor(buffer, dtype=torch.int16)
            tmp = f"{CHUNK_DIR}/chunk_{chunk_idx:04d}.pt.tmp"
            torch.save(arr, tmp)
            os.replace(tmp, f"{CHUNK_DIR}/chunk_{chunk_idx:04d}.pt")
            print(f"  chunk {chunk_idx:03d}: {len(buffer)/1e6:.0f}M tokens | {examples_done:,} ex")
            buffer = buffer[-BLOCK_SIZE:]
            chunk_idx += 1
            if chunk_idx >= MAX_CHUNKS:
                break

    if buffer and chunk_idx < MAX_CHUNKS:
        arr = torch.tensor(buffer, dtype=torch.int16)
        tmp = f"{CHUNK_DIR}/chunk_{chunk_idx:04d}.pt.tmp"
        torch.save(arr, tmp)
        os.replace(tmp, f"{CHUNK_DIR}/chunk_{chunk_idx:04d}.pt")
        print(f"  final chunk {chunk_idx:03d}: {len(buffer)/1e6:.0f}M tokens")

print("Chunks:", len(glob.glob(f"{CHUNK_DIR}/chunk_*.pt")))

Existing: 100 | Target: 100
Chunks: 100


In [5]:
# =========================================================
# CELL 5: Model classes (100M, RoPE + SwiGLU + GQA + RMSNorm)
# =========================================================
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        x_fp32 = x.float()
        norm = torch.rsqrt(x_fp32.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x_fp32 * norm).to(x.dtype) * self.weight


class SwiGLU(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        hidden = int(8 * d_model / 3)
        hidden = 64 * ((hidden + 63) // 64)
        self.w1 = nn.Linear(d_model, hidden, bias=False)
        self.w2 = nn.Linear(hidden, d_model, bias=False)
        self.w3 = nn.Linear(d_model, hidden, bias=False)
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class Attention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads    = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim   = d_model // n_heads
        self.n_rep      = n_heads // n_kv_heads
        self.wq = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, freqs_cis):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.head_dim)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.head_dim)

        # RoPE inline
        q_c = torch.view_as_complex(q.float().reshape(*q.shape[:-1], -1, 2).contiguous())
        k_c = torch.view_as_complex(k.float().reshape(*k.shape[:-1], -1, 2).contiguous())
        f   = freqs_cis[None, :, None, :].to(q_c.dtype)
        q = torch.view_as_real(q_c * f).flatten(-2).to(x.dtype)
        k = torch.view_as_real(k_c * f).flatten(-2).to(x.dtype)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        if self.n_heads != self.n_kv_heads:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.wo(out)


class Block(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        self.ln1  = RMSNorm(d_model)
        self.attn = Attention(d_model, n_heads, n_kv_heads)
        self.ln2  = RMSNorm(d_model)
        self.mlp  = SwiGLU(d_model)
    def forward(self, x, freqs_cis):
        x = x + self.attn(self.ln1(x), freqs_cis)
        x = x + self.mlp(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=768, n_heads=12, n_kv_heads=4,
                 n_layers=12, max_len=1024, rope_theta=10000.0):
        super().__init__()
        self.d_model  = d_model
        self.n_layers = n_layers
        self.max_len  = max_len
        self.head_dim = d_model // n_heads

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks  = nn.ModuleList([
            Block(d_model, n_heads, n_kv_heads) for _ in range(n_layers)
        ])
        self.ln_f = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if isinstance(m, nn.Linear) and m.bias is not None:
                    nn.init.zeros_(m.bias)

        self.head.weight = self.tok_emb.weight  # weight tying

        scale = 1.0 / math.sqrt(2 * n_layers)
        with torch.no_grad():
            for block in self.blocks:
                block.attn.wo.weight.mul_(scale)
                block.mlp.w2.weight.mul_(scale)

        d = self.head_dim
        freqs     = 1.0 / (rope_theta ** (torch.arange(0, d, 2, dtype=torch.float32) / d))
        positions = torch.arange(max_len, dtype=torch.float32)
        angles    = torch.outer(positions, freqs)
        self.register_buffer("freqs_cis",
                             torch.polar(torch.ones_like(angles), angles),
                             persistent=False)

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx)
        freqs = self.freqs_cis[:T]
        for block in self.blocks:
            x = block(x, freqs)
        return self.head(self.ln_f(x))

print("Model classes ready")

Model classes ready


In [6]:
# =========================================================
# CELL 6: Config + Model init
# =========================================================
D_MODEL    = 768
N_HEADS    = 12
N_KV_HEADS = 4
N_LAYERS   = 12
MAX_LEN    = 1024

model = MiniGPT(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_kv_heads=N_KV_HEADS,
    n_layers=N_LAYERS,
    max_len=MAX_LEN,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Params: {n_params/1e6:.2f}M")
assert 90e6 < n_params < 115e6, "Should be ~100M"

Params: 100.09M


In [7]:
# =========================================================
# CELL 7: Optimizer + Scheduler + Resume (auto)
# =========================================================
MICRO_BATCH = 8
GRAD_ACCUM  = 4
BLOCK_SIZE  = 1024

LR           = 3e-4
MIN_LR       = 3e-5
WARMUP_STEPS = 1000

TOTAL_TOKENS    = 3_000_000_000
TOKENS_PER_STEP = MICRO_BATCH * GRAD_ACCUM * BLOCK_SIZE   # 32768
TOTAL_STEPS     = TOTAL_TOKENS // TOKENS_PER_STEP         # ~91553

CKPT_EVERY   = 200
LOG_EVERY    = 50
EVAL_EVERY   = 1000
EVAL_BATCHES = 20

print(f"Tokens/step: {TOKENS_PER_STEP:,} | Total steps: {TOTAL_STEPS:,}")

# ---------- Optimizer ----------
decay_params, no_decay_params = [], []
for name, p in model.named_parameters():
    if not p.requires_grad: continue
    (decay_params if p.dim() >= 2 else no_decay_params).append(p)

optimizer = torch.optim.AdamW([
    {"params": decay_params,    "weight_decay": 0.1},
    {"params": no_decay_params, "weight_decay": 0.0},
], lr=LR, betas=(0.9, 0.95))

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lambda step: (
        (step + 1) / WARMUP_STEPS if step < WARMUP_STEPS
        else (MIN_LR/LR if step >= TOTAL_STEPS
              else (MIN_LR + (LR-MIN_LR) * 0.5 * (1+math.cos(math.pi*(step-WARMUP_STEPS)/(TOTAL_STEPS-WARMUP_STEPS)))) / LR)
    )
)

# ---------- Resume ----------
start_step = 0
best_loss  = float("inf")

if os.path.exists(LATEST_PATH):
    ckpt = None
    for p in [LATEST_PATH, LATEST_PATH + ".bak"]:
        if not os.path.exists(p): continue
        try:
            ckpt = torch.load(p, map_location=device, weights_only=False)
            print(f"Loaded: {p}")
            break
        except Exception as e:
            print(f"Skip {p}: {e}")
    if ckpt is not None:
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        start_step = ckpt["global_step"]
        best_loss  = ckpt.get("best_loss", float("inf"))
        print(f"✅ Resumed: step {start_step} | best: {best_loss:.4f}")
    else:
        print("🆕 All corrupt — fresh start")
else:
    print("🆕 Fresh training")

print(f"Starting from step {start_step} / {TOTAL_STEPS}")

Tokens/step: 32,768 | Total steps: 91,552
Loaded: /content/drive/MyDrive/gpt100m/checkpoints/latest.pth
✅ Resumed: step 20800 | best: 3.2872
Starting from step 20800 / 91552


In [8]:
# =========================================================
# CELL 8: Sanity check (loss ~ log(VOCAB) at start)
# =========================================================
model.train()

test_chunk = torch.load(sorted(glob.glob(f"{CHUNK_DIR}/chunk_*.pt"))[0],
                        weights_only=True).long()

ix = torch.randint(0, len(test_chunk) - BLOCK_SIZE - 1, (2,))
x = torch.stack([test_chunk[i:i+BLOCK_SIZE] for i in ix]).to(device)
y = torch.stack([test_chunk[i+1:i+BLOCK_SIZE+1] for i in ix]).to(device)

with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))

print(f"Logits: {tuple(logits.shape)}")
print(f"Loss: {loss.item():.3f} | Expected ~{math.log(VOCAB_SIZE):.3f}")

if start_step == 0 and abs(loss.item() - math.log(VOCAB_SIZE)) > 0.5:
    print("⚠️ Init loss off — check model!")
else:
    print("✅ Sanity OK")

del test_chunk, x, y, logits, loss
gc.collect()
torch.cuda.empty_cache()

Logits: (2, 1024, 32000)
Loss: 3.307 | Expected ~10.373
✅ Sanity OK


In [ ]:
# =========================================================
# CELL 9: PRETRAINING LOOP (base model yahan banta hai)
# auto-resume, atomic checkpoint, best save
# =========================================================
model.train()
chunk_files = sorted(glob.glob(f"{CHUNK_DIR}/chunk_*.pt"))
print(f"Training on {len(chunk_files)} chunks")

current_chunk   = None
chunk_load_step = -1
session_start   = time.time()

try:
    for global_step in range(start_step, TOTAL_STEPS):

        # ---------- Fresh chunk every 100 steps ----------
        if global_step - chunk_load_step >= 100 or current_chunk is None:
            f = random.choice(chunk_files)
            current_chunk = torch.load(f, weights_only=True).long().to(device)
            chunk_load_step = global_step

        # ---------- Gradient accumulation ----------
        optimizer.zero_grad(set_to_none=True)
        accum_loss = 0.0

        for _ in range(GRAD_ACCUM):
            ix = torch.randint(0, len(current_chunk) - BLOCK_SIZE - 1, (MICRO_BATCH,))
            x = torch.stack([current_chunk[i:i+BLOCK_SIZE] for i in ix])
            y = torch.stack([current_chunk[i+1:i+BLOCK_SIZE+1] for i in ix])

            with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = model(x)
                raw_loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
                loss = raw_loss / GRAD_ACCUM

            scaler.scale(loss).backward()
            accum_loss += raw_loss.item()

        scaler.unscale_(optimizer)
        clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        avg_loss = accum_loss / GRAD_ACCUM

        # ---------- Log ----------
        if global_step % LOG_EVERY == 0:
            elapsed = time.time() - session_start
            cur_lr = optimizer.param_groups[0]["lr"]
            print(f"  step {global_step:6d} | loss {avg_loss:.4f} | lr {cur_lr:.2e} | {elapsed/60:.1f} min")

        # ---------- Checkpoint (atomic) ----------
        if global_step % CKPT_EVERY == 0 and global_step > 0:
            if os.path.exists(LATEST_PATH):
                try: os.replace(LATEST_PATH, LATEST_PATH + ".bak")
                except Exception: pass
            tmp = LATEST_PATH + ".tmp"
            torch.save({
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "global_step": global_step,
                "best_loss": best_loss,
            }, tmp)
            os.replace(tmp, LATEST_PATH)
            print(f"  💾 ckpt @ step {global_step}")

        # ---------- Eval + save best ----------
        if global_step % EVAL_EVERY == 0 and global_step > 0:
            model.eval()
            val_chunk = torch.load(random.choice(chunk_files),
                                   weights_only=True).long().to(device)
            total_val = 0.0
            with torch.no_grad():
                for _ in range(EVAL_BATCHES):
                    i = random.randint(0, len(val_chunk) - BLOCK_SIZE - 1)
                    vx = val_chunk[i:i+BLOCK_SIZE].unsqueeze(0)
                    vy = val_chunk[i+1:i+BLOCK_SIZE+1].unsqueeze(0)
                    with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                        vl = model(vx)
                        vloss = F.cross_entropy(vl.view(-1, vl.size(-1)), vy.view(-1))
                    total_val += vloss.item()
            avg_val = total_val / EVAL_BATCHES
            print(f"  📊 VAL loss: {avg_val:.4f}")

            if avg_val < best_loss:
                best_loss = avg_val
                if os.path.exists(BEST_PATH):
                    try: os.replace(BEST_PATH, BEST_PATH + ".bak")
                    except Exception: pass
                tmp = BEST_PATH + ".tmp"
                torch.save({
                    "model": model.state_dict(),
                    "global_step": global_step,
                    "best_loss": best_loss,
                }, tmp)
                os.replace(tmp, BEST_PATH)
                print(f"  ✅ NEW BEST: {best_loss:.4f}")

            del val_chunk
            model.train()

        if global_step % 500 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    print("\n🎉 Pretraining complete! Base model ready.")

except KeyboardInterrupt:
    print(f"\n⏹️ Interrupted at step {global_step}")
    if os.path.exists(LATEST_PATH):
        try: os.replace(LATEST_PATH, LATEST_PATH + ".bak")
        except Exception: pass
    tmp = LATEST_PATH + ".tmp"
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "global_step": global_step,
        "best_loss": best_loss,
    }, tmp)
    os.replace(tmp, LATEST_PATH)
    print(f"✅ Saved at step {global_step}. Next session resume karega.")

Training on 100 chunks
  step  20800 | loss 3.2768 | lr 2.69e-04 | 0.6 min
  💾 ckpt @ step 20800
  step  20850 | loss 3.2334 | lr 2.69e-04 | 2.3 min
  step  20900 | loss 3.2956 | lr 2.69e-04 | 4.1 min
  step  20950 | loss 3.2938 | lr 2.69e-04 | 5.9 min
  step  21000 | loss 3.3404 | lr 2.69e-04 | 7.7 min
  💾 ckpt @ step 21000
  📊 VAL loss: 3.3872
  step  21050 | loss 3.3957 | lr 2.69e-04 | 9.5 min
  step  21100 | loss 3.4384 | lr 2.68e-04 | 11.4 min
  step  21150 | loss 3.4960 | lr 2.68e-04 | 13.1 min
  step  21200 | loss 3.4948 | lr 2.68e-04 | 14.9 min
  💾 ckpt @ step 21200
  step  21250 | loss 3.4365 | lr 2.68e-04 | 16.8 min
  step  21300 | loss 3.3991 | lr 2.68e-04 | 18.6 min
  step  21350 | loss 3.5731 | lr 2.68e-04 | 20.3 min
  step  21400 | loss 3.3176 | lr 2.68e-04 | 22.1 min
  💾 ckpt @ step 21400
  step  21450 | loss 3.4713 | lr 2.67e-04 | 24.0 min
  step  21500 | loss 3.5601 | lr 2.67e-04 | 25.8 min
  step  21550 | loss 3.3897 | lr 2.67e-04 | 27.6 min
  step  21600 | loss 3.423

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =========================================================
# CELL 10: COMPLETE TESTING CELL (standalone)
# Naye Colab session me bhi chalega — sirf Drive mount karo
# =========================================================

# ---------- 1. Mount Drive ----------
from google.colab import drive
drive.mount('/content/drive')

# ---------- 2. Imports ----------
import os, math, random, glob
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer

# ---------- 3. Paths ----------
BASE_DIR   = "/content/drive/MyDrive/gpt100m"
CHUNK_DIR  = f"{BASE_DIR}/chunks"
CKPT_DIR   = f"{BASE_DIR}/checkpoints"
TOK_PATH   = f"{BASE_DIR}/tokenizer_32k.json"
LATEST_PATH = f"{CKPT_DIR}/latest.pth"
BEST_PATH   = f"{CKPT_DIR}/best.pth"

# ---------- 4. Device + AMP ----------
device = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.float16
print("Device:", device)

# ---------- 5. Config (pretraining ke saath match karna zaroori) ----------
D_MODEL    = 768
N_HEADS    = 12
N_KV_HEADS = 4
N_LAYERS   = 12
MAX_LEN    = 1024

# ---------- 6. Model classes (CELL 5 se copy) ----------
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        x_fp32 = x.float()
        norm = torch.rsqrt(x_fp32.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x_fp32 * norm).to(x.dtype) * self.weight

class SwiGLU(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        hidden = int(8 * d_model / 3)
        hidden = 64 * ((hidden + 63) // 64)
        self.w1 = nn.Linear(d_model, hidden, bias=False)
        self.w2 = nn.Linear(hidden, d_model, bias=False)
        self.w3 = nn.Linear(d_model, hidden, bias=False)
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class Attention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads    = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim   = d_model // n_heads
        self.n_rep      = n_heads // n_kv_heads
        self.wq = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x, freqs_cis):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.head_dim)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.head_dim)
        q_c = torch.view_as_complex(q.float().reshape(*q.shape[:-1], -1, 2).contiguous())
        k_c = torch.view_as_complex(k.float().reshape(*k.shape[:-1], -1, 2).contiguous())
        f   = freqs_cis[None, :, None, :].to(q_c.dtype)
        q = torch.view_as_real(q_c * f).flatten(-2).to(x.dtype)
        k = torch.view_as_real(k_c * f).flatten(-2).to(x.dtype)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        if self.n_heads != self.n_kv_heads:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.wo(out)

class Block(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        self.ln1  = RMSNorm(d_model)
        self.attn = Attention(d_model, n_heads, n_kv_heads)
        self.ln2  = RMSNorm(d_model)
        self.mlp  = SwiGLU(d_model)
    def forward(self, x, freqs_cis):
        x = x + self.attn(self.ln1(x), freqs_cis)
        x = x + self.mlp(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=768, n_heads=12, n_kv_heads=4,
                 n_layers=12, max_len=1024, rope_theta=10000.0):
        super().__init__()
        self.d_model  = d_model
        self.n_layers = n_layers
        self.max_len  = max_len
        self.head_dim = d_model // n_heads
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks  = nn.ModuleList([
            Block(d_model, n_heads, n_kv_heads) for _ in range(n_layers)
        ])
        self.ln_f = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if isinstance(m, nn.Linear) and m.bias is not None:
                    nn.init.zeros_(m.bias)
        self.head.weight = self.tok_emb.weight
        scale = 1.0 / math.sqrt(2 * n_layers)
        with torch.no_grad():
            for block in self.blocks:
                block.attn.wo.weight.mul_(scale)
                block.mlp.w2.weight.mul_(scale)
        d = self.head_dim
        freqs     = 1.0 / (rope_theta ** (torch.arange(0, d, 2, dtype=torch.float32) / d))
        positions = torch.arange(max_len, dtype=torch.float32)
        angles    = torch.outer(positions, freqs)
        self.register_buffer("freqs_cis",
                             torch.polar(torch.ones_like(angles), angles),
                             persistent=False)
    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx)
        freqs = self.freqs_cis[:T]
        for block in self.blocks:
            x = block(x, freqs)
        return self.head(self.ln_f(x))

# ---------- 7. Load tokenizer ----------
tokenizer = Tokenizer.from_file(TOK_PATH)
VOCAB_SIZE = tokenizer.get_vocab_size()
END_ID     = tokenizer.token_to_id("<|end|>")
PAD_ID     = tokenizer.token_to_id("<pad>")
print(f"Tokenizer loaded | Vocab: {VOCAB_SIZE}")

# ---------- 8. Model init ----------
model = MiniGPT(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_kv_heads=N_KV_HEADS,
    n_layers=N_LAYERS,
    max_len=MAX_LEN,
).to(device)

print(f"Model params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

# ---------- 9. Load checkpoint ----------
# Yahan choose karo: "latest" ya "best"
LOAD_WHICH = "best"   # "latest" | "best"

ckpt = None
if LOAD_WHICH == "best" and os.path.exists(BEST_PATH):
    ckpt = torch.load(BEST_PATH, map_location=device, weights_only=False)
    print(f"Loaded BEST | step {ckpt.get('global_step', '?')} | val {ckpt.get('best_loss', 0):.4f}")
elif os.path.exists(LATEST_PATH):
    ckpt = torch.load(LATEST_PATH, map_location=device, weights_only=False)
    print(f"Loaded LATEST | step {ckpt.get('global_step', '?')} | best {ckpt.get('best_loss', 0):.4f}")
else:
    # Fallback: try .bak
    for p in [LATEST_PATH + ".bak", BEST_PATH + ".bak"]:
        if os.path.exists(p):
            try:
                ckpt = torch.load(p, map_location=device, weights_only=False)
                print(f"Loaded fallback: {p}")
                break
            except Exception:
                pass

if ckpt is not None:
    model.load_state_dict(ckpt["model"])
else:
    print("⚠️ No checkpoint found! Using random weights (expect garbage).")

model.eval()
print("Model ready for generation ----")

# =========================================================
# 10. GENERATION CONTROLS
# =========================================================
PROMPT             = 'Anmol Chaudhary loves to Alisha'
MAX_TOKENS         = 150
TEMPERATURE        = 0.7     # 0.1 = safe, 1.0 = normal, 1.5 = chaos
TOP_P              = 0.9      # 0.5 = safe, 0.95 = diverse, 1.0 = off
TOP_K              = 0        # 0 = off, 40 = top 40 only
REPETITION_PENALTY = 1.1      # 1.0 = off, 1.2 = strong
SEED               = None     # None = random, 42 = reproducible

# =========================================================
# 11. GENERATE
# =========================================================
if SEED is not None:
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

ids = tokenizer.encode(PROMPT).ids
out_ids = []
recent_tokens = []

with torch.no_grad():
    for _ in range(MAX_TOKENS):
        window = ids[-MAX_LEN:]
        x = torch.tensor([window], dtype=torch.long, device=device)

        with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(x)

        next_logits = logits[0, -1, :].float()

        # Repetition penalty
        if REPETITION_PENALTY != 1.0 and recent_tokens:
            for tok in set(recent_tokens[-50:]):
                if next_logits[tok] > 0:
                    next_logits[tok] /= REPETITION_PENALTY
                else:
                    next_logits[tok] *= REPETITION_PENALTY

        # Temperature
        if TEMPERATURE != 1.0:
            next_logits = next_logits / max(TEMPERATURE, 1e-5)

        # Top-K
        if TOP_K > 0:
            topk_vals, _ = torch.topk(next_logits, TOP_K)
            next_logits[next_logits < topk_vals[-1]] = float("-inf")

        # Top-P + sample
        probs = F.softmax(next_logits, dim=-1)
        if TOP_P < 1.0:
            sorted_p, sorted_i = torch.sort(probs, descending=True)
            cum = torch.cumsum(sorted_p, dim=-1)
            remove = cum > TOP_P
            remove[1:] = remove[:-1].clone()
            remove[0] = False
            sorted_p[remove] = 0.0
            sorted_p = sorted_p / sorted_p.sum()
            next_id = sorted_i[torch.multinomial(sorted_p, 1)].item()
        else:
            next_id = torch.multinomial(probs, 1).item()

        if next_id == END_ID:
            break

        ids.append(next_id)
        out_ids.append(next_id)
        recent_tokens.append(next_id)

# =========================================================
# 12. PRINT
# =========================================================
print("\n" + "=" * 60)
print(f"PROMPT: {PROMPT}")
print(f"T={TEMPERATURE}, top_p={TOP_P}, top_k={TOP_K}, rep_pen={REPETITION_PENALTY}")
print("=" * 60)
print(tokenizer.decode(out_ids))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cpu
Tokenizer loaded | Vocab: 32000
Model params: 100.09M
Loaded BEST | step 17000 | val 3.2872
Model ready for generation ----

PROMPT: Anmol Chaudhary loves to Alisha
T=0.7, top_p=0.9, top_k=0, rep_pen=1.1
, the second-born daughter of Alisha. Alisha is a bright child, and she is very proud of her family.
Allah Chaudhary wants to hear from everyone who will hear her. He says that Alisha must be alive, even if she has no family or friends. He says that Alisha has to be strong and brave. Aliyama does not have a family, but he is good. Aliyama does not know how to be tough and brave. He says that Aliyama has a good heart, and he can do everything with his heart. He says that Aliyama is weak, but she is strong and courageous. He is kind and gentle, and the story is a book. He
